# Model Experiment — `Prophet`


## Setup — install dependencies (Kaggle only; local venv already has these via requirements-models.txt)

In [ ]:
import pathlib
if pathlib.Path('/kaggle/input').exists():
    !pip install -q prophet wandb python-dotenv mlflow

## 0. Config


In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_COMPETITION = 'walmart-recruiting-store-sales-forecasting'
ON_KAGGLE = KAGGLE_INPUT.exists()

if ON_KAGGLE:
    WORKING_DIR = Path('/kaggle/working')
else:
    PROJECT_ROOT = Path.cwd().parent
    WORKING_DIR = PROJECT_ROOT
    load_dotenv(PROJECT_ROOT / '.env')

def _find_file(stem):

    names = (f'{stem}.csv', f'{stem}.csv.zip')
    if ON_KAGGLE:
        search_roots = [
            KAGGLE_INPUT / KAGGLE_COMPETITION,
            KAGGLE_INPUT / 'competitions' / KAGGLE_COMPETITION,
        ]
        for root in search_roots:
            if root.exists():
                for name in names:
                    p = root / name
                    if p.exists():
                        return p
        for name in names:
            matches = list(KAGGLE_INPUT.rglob(name))
            if matches:
                return matches[0]
        raise FileNotFoundError(
            f"Could not find {stem}.csv[.zip] anywhere under {KAGGLE_INPUT}. "
            f"Contents of {KAGGLE_INPUT}: {list(KAGGLE_INPUT.iterdir())}"
        )
    else:
        p = PROJECT_ROOT / 'data' / 'raw' / f'{stem}.csv'
        if not p.exists():
            raise FileNotFoundError(f'{p} not found')
        return p

TRAIN_CSV = _find_file('train')
TEST_CSV = _find_file('test')
FEATURES_CSV = _find_file('features')
STORES_CSV = _find_file('stores')

RANDOM_SEED = 42
TARGET = 'Weekly_Sales'
HOLIDAY_WEIGHT = 5
NON_HOLIDAY_WEIGHT = 1

if ON_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
        for key in ('MLFLOW_TRACKING_URI', 'MLFLOW_TRACKING_USERNAME', 'MLFLOW_TRACKING_PASSWORD', 'WANDB_API_KEY'):
            try:
                os.environ.setdefault(key, client.get_secret(key))
            except Exception:
                pass
    except Exception:
        pass

MLFLOW_TRACKING_URI = os.getenv('MLFLOW_TRACKING_URI')
MLFLOW_TRACKING_USERNAME = os.getenv('MLFLOW_TRACKING_USERNAME')
MLFLOW_TRACKING_PASSWORD = os.getenv('MLFLOW_TRACKING_PASSWORD')

print('On Kaggle:', ON_KAGGLE)
print('train:', TRAIN_CSV)
print('test:', TEST_CSV)
print('features:', FEATURES_CSV)
print('stores:', STORES_CSV)

## 1. Data loading helpers

In [ ]:
import pandas as pd

def _read_bool(series):
    if series.dtype == bool:
        return series
    return series.astype(str).str.strip().str.upper().map({'TRUE': True, 'FALSE': False})

def load_stores():
    return pd.read_csv(STORES_CSV)

def load_features():
    df = pd.read_csv(FEATURES_CSV, parse_dates=['Date'])
    df['IsHoliday'] = _read_bool(df['IsHoliday'])
    df = df.sort_values(['Store', 'Date'])
    for col in ('CPI', 'Unemployment'):
        df[col] = df.groupby('Store')[col].transform(lambda s: s.ffill().bfill())
    return df.reset_index(drop=True)

def load_raw(split):
    path = TRAIN_CSV if split == 'train' else TEST_CSV
    df = pd.read_csv(path, parse_dates=['Date'])
    df['IsHoliday'] = _read_bool(df['IsHoliday'])
    return df

def load_merged(split='train'):
    base = load_raw(split)
    stores = load_stores()
    feats = load_features().drop(columns=['IsHoliday'])
    df = base.merge(stores, on='Store', how='left')
    df = df.merge(feats, on=['Store', 'Date'], how='left')
    df = df.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)
    return df

## 2. Metric (WMAE) & cross-validation helpers

In [ ]:
def weights_from_holiday(is_holiday):
    is_holiday = np.asarray(is_holiday).astype(bool)
    return np.where(is_holiday, HOLIDAY_WEIGHT, NON_HOLIDAY_WEIGHT).astype(float)

def wmae(y_true, y_pred, is_holiday):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    w = weights_from_holiday(is_holiday)
    return float(np.sum(w * np.abs(y_true - y_pred)) / np.sum(w))

In [ ]:
def _sorted_unique_dates(dates):
    return np.sort(pd.to_datetime(dates).unique())

def time_holdout(df, weeks=8, date_col='Date'):
    uniq = _sorted_unique_dates(df[date_col])
    if weeks >= len(uniq):
        raise ValueError(f'weeks={weeks} >= number of distinct weeks {len(uniq)}')
    cutoff = uniq[-weeks]
    d = pd.to_datetime(df[date_col]).to_numpy()
    train_idx = np.where(d < cutoff)[0]
    val_idx = np.where(d >= cutoff)[0]
    return train_idx, val_idx

def expanding_splits(df, n_splits=3, horizon=8, date_col='Date'):
    uniq = _sorted_unique_dates(df[date_col])
    needed = horizon * n_splits
    if needed >= len(uniq):
        raise ValueError(f'Need > {needed} distinct weeks for {n_splits} folds of {horizon}; have {len(uniq)}.')
    d = pd.to_datetime(df[date_col]).to_numpy()
    for k in range(n_splits):
        end_offset = needed - k * horizon
        start_offset = end_offset - horizon
        val_start = uniq[-end_offset]
        val_end = uniq[-start_offset] if start_offset > 0 else None
        train_idx = np.where(d < val_start)[0]
        if val_end is None:
            val_idx = np.where(d >= val_start)[0]
        else:
            val_idx = np.where((d >= val_start) & (d < val_end))[0]
        yield train_idx, val_idx

## 3. Holiday effects & shared helpers


In [1]:
SUPER_BOWL = pd.to_datetime(['2010-02-12', '2011-02-11', '2012-02-10', '2013-02-08'])
LABOR_DAY = pd.to_datetime(['2010-09-10', '2011-09-09', '2012-09-07', '2013-09-06'])
THANKSGIVING = pd.to_datetime(['2010-11-26', '2011-11-25', '2012-11-23', '2013-11-29'])
CHRISTMAS = pd.to_datetime(['2010-12-31', '2011-12-30', '2012-12-28', '2013-12-27'])

def build_holidays_df():
    frames = []
    for name, dates in [('SuperBowl', SUPER_BOWL), ('LaborDay', LABOR_DAY),
                         ('Thanksgiving', THANKSGIVING), ('Christmas', CHRISTMAS)]:
        frames.append(pd.DataFrame({'holiday': name, 'ds': dates, 'lower_window': 0, 'upper_window': 0}))
    return pd.concat(frames, ignore_index=True)

HOLIDAYS_DF = build_holidays_df()

def cold_start_fallback_table(df, value_col='Weekly_Sales'):

    fallback_per_key = df.groupby(['Store', 'Dept'])[value_col].apply(lambda s: s.tail(8).mean())
    dept_mean = df.groupby('Dept')[value_col].mean()
    global_mean = float(df[value_col].mean())
    return fallback_per_key, dept_mean, global_mean

NameError: name 'pd' is not defined

## 4. Weights & Biases setup

In [ ]:
import wandb
import numpy as np, pandas as pd

MODEL_NAME = 'Prophet'
WANDB_PROJECT = 'walmart-sales-forecasting'
wandb.login()

## 5. MLflow tracking setup


In [ ]:
import mlflow

def init_tracking(experiment=None):
    uri = MLFLOW_TRACKING_URI
    if uri:

        if 'dagshub.com' in uri and not uri.rstrip('/').endswith('.mlflow'):
            uri = uri.rstrip('/') + '.mlflow'
            print(f'NOTE: appended .mlflow to MLFLOW_TRACKING_URI (DagsHub requires it) -> {uri}')
        mlflow.set_tracking_uri(uri)
        if MLFLOW_TRACKING_USERNAME:
            os.environ['MLFLOW_TRACKING_USERNAME'] = MLFLOW_TRACKING_USERNAME
        if MLFLOW_TRACKING_PASSWORD:
            os.environ['MLFLOW_TRACKING_PASSWORD'] = MLFLOW_TRACKING_PASSWORD
    else:

        print('WARNING: MLFLOW_TRACKING_URI not set -- check Add-ons > Secrets is attached to THIS notebook.')
        print('Falling back to a LOCAL sqlite store: this will NOT persist beyond this session and your team will not see it.')
        mlflow.set_tracking_uri(f"sqlite:///{WORKING_DIR / 'mlflow.db'}")
    if experiment:
        mlflow.set_experiment(experiment)
    return mlflow.get_tracking_uri()

EXPERIMENT = f'{MODEL_NAME}_Training'
mlflow_uri = init_tracking(EXPERIMENT)
print('MLflow ->', mlflow_uri, '| experiment:', EXPERIMENT)


## Load raw data

In [ ]:
raw_train = load_raw('train')
raw_test  = load_raw('test')
raw_train.shape, raw_test.shape

## Run 1 — Cleaning


In [ ]:
n_series = raw_train.groupby(['Store', 'Dept']).ngroups
lengths = raw_train.groupby(['Store', 'Dept']).size()
span_weeks = raw_train.groupby(['Store', 'Dept'])['Date'].agg(lambda d: (d.max() - d.min()).days // 7 + 1)
n_with_gaps = int((lengths < span_weeks).sum())
n_negative_sales = int((raw_train['Weekly_Sales'] < 0).sum())

cleaning_stats = {
    'n_train_rows': len(raw_train),
    'n_series': n_series,
    'n_series_with_gaps': n_with_gaps,
    'n_negative_sales': n_negative_sales,
}

with mlflow.start_run(run_name=f'{MODEL_NAME}_Cleaning'):
    for k, v in cleaning_stats.items():
        mlflow.log_metric(k, v)

run = wandb.init(project=WANDB_PROJECT, group=MODEL_NAME, job_type='cleaning', name=f'{MODEL_NAME}_Cleaning')
wandb.log(cleaning_stats)
run.finish()
print(f'{n_series} series, {n_with_gaps} with missing weeks, {n_negative_sales} negative Weekly_Sales rows')

## Run 2 — "Feature Selection" (informational)


In [ ]:
feature_selection_decisions = {
    'holidays_used': 'SuperBowl, LaborDay, Thanksgiving, Christmas (native Prophet holiday-effect term)',
    'extra_regressors_used': 'none (add_regressor deliberately not used, see Theory)',
    'weekly_seasonality': False,
    'yearly_seasonality': True,
    'reasoning': 'per-series fits keep exogenous regressors fragile/unnecessary here; '
                 'LightGBM.ipynb/XGBoost.ipynb already cover that signal with a global model',
}

with mlflow.start_run(run_name=f'{MODEL_NAME}_Feature_Selection'):
    for k, v in feature_selection_decisions.items():
        mlflow.log_param(k, v)

run = wandb.init(project=WANDB_PROJECT, group=MODEL_NAME, job_type='feature_selection', name=f'{MODEL_NAME}_Feature_Selection')
wandb.config.update(feature_selection_decisions)
run.finish()
for k, v in feature_selection_decisions.items():
    print(f'{k}: {v}')

## 6. Pipeline wrapper (raw-test-ready)


In [2]:
from sklearn.base import BaseEstimator, RegressorMixin
import logging
logging.getLogger('cmdstanpy').disabled = True
logging.getLogger('cmdstanpy').setLevel(logging.CRITICAL)
logging.getLogger('prophet').setLevel(logging.WARNING)
from prophet import Prophet

class PerSeriesProphetForecaster(BaseEstimator, RegressorMixin):


    def __init__(self, changepoint_prior_scale=0.05, seasonality_prior_scale=10.0,
                 seasonality_mode='additive', holidays_prior_scale=10.0, min_obs=20):
        self.changepoint_prior_scale = changepoint_prior_scale
        self.seasonality_prior_scale = seasonality_prior_scale
        self.seasonality_mode = seasonality_mode
        self.holidays_prior_scale = holidays_prior_scale
        self.min_obs = min_obs

    def fit(self, X, y=None):
        df = X.copy()
        df['Weekly_Sales'] = np.asarray(y, dtype=float)

        self.models_ = {}
        self.last_date_ = {}
        fallback_per_key, self.dept_mean_, self.global_mean_ = cold_start_fallback_table(df)
        self.fallback_value_ = fallback_per_key.to_dict()

        n_failed = 0
        for key, g in df.groupby(['Store', 'Dept']):
            g = g.sort_values('Date').drop_duplicates('Date')
            s = g.set_index('Date')['Weekly_Sales']
            self.last_date_[key] = s.index.max()
            if len(s) < self.min_obs:
                self.models_[key] = None
                continue
            try:
                pdf = pd.DataFrame({'ds': s.index, 'y': s.to_numpy()})
                m = Prophet(
                    changepoint_prior_scale=self.changepoint_prior_scale,
                    seasonality_prior_scale=self.seasonality_prior_scale,
                    seasonality_mode=self.seasonality_mode,
                    holidays_prior_scale=self.holidays_prior_scale,
                    yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False,
                    holidays=HOLIDAYS_DF,
                )
                m.fit(pdf)
                self.models_[key] = m
            except Exception:
                self.models_[key] = None
                n_failed += 1

        self.n_series_ = df.groupby(['Store', 'Dept']).ngroups
        self.n_failed_ = n_failed
        return self

    def _fallback(self, key, dept):
        if key in self.fallback_value_:
            return self.fallback_value_[key]
        if dept in self.dept_mean_.index:
            return float(self.dept_mean_.loc[dept])
        return self.global_mean_

    def predict(self, X):
        preds = np.empty(len(X), dtype=float)
        X = X.reset_index(drop=True)
        for (store, dept), idx in X.groupby(['Store', 'Dept']).groups.items():
            key = (store, dept)
            rows = X.loc[idx]
            if key not in self.models_ or self.models_[key] is None:
                preds[idx] = self._fallback(key, dept)
                continue
            future = pd.DataFrame({'ds': pd.to_datetime(rows['Date'])})
            try:
                fc = self.models_[key].predict(future)
                preds[idx] = fc['yhat'].to_numpy()
            except Exception:
                preds[idx] = self._fallback(key, dept)
        return np.clip(preds, 0, None)

## Run 3 — Cross-validation (curated hyperparameter search)


In [ ]:
import itertools

SAMPLE_SIZE = 150
rng = np.random.RandomState(RANDOM_SEED)
all_keys = list(raw_train.groupby(['Store', 'Dept']).groups.keys())
sample_keys = [all_keys[i] for i in rng.choice(len(all_keys), size=min(SAMPLE_SIZE, len(all_keys)), replace=False)]
sample_mask = raw_train.set_index(['Store', 'Dept']).index.isin(sample_keys)
sample_df = raw_train[sample_mask].reset_index(drop=True)
print('CV sample:', len(sample_keys), 'series,', len(sample_df), 'rows')

candidate_changepoint_prior_scale = [0.01, 0.05, 0.1, 0.5]
candidate_seasonality_prior_scale = [1.0, 10.0, 25.0]
candidate_seasonality_mode = ['additive', 'multiplicative']
candidate_holidays_prior_scale = [1.0, 10.0]

configs = [
    {'changepoint_prior_scale': cp, 'seasonality_prior_scale': sp, 'seasonality_mode': sm, 'holidays_prior_scale': hp}
    for cp, sp, sm, hp in itertools.product(
        candidate_changepoint_prior_scale, candidate_seasonality_prior_scale,
        candidate_seasonality_mode, candidate_holidays_prior_scale,
    )
]
print(len(configs), 'configs to try')

trial_results = []
with mlflow.start_run(run_name=f'{MODEL_NAME}_CV'):
    mlflow.log_param('n_configs', len(configs))
    mlflow.log_param('n_folds', 3)
    mlflow.log_param('sample_size_series', len(sample_keys))
    for i, cfg in enumerate(configs):
        run = wandb.init(project=WANDB_PROJECT, group=MODEL_NAME, job_type='cv', name=f'{MODEL_NAME}_CV_trial{i}', config=cfg, reinit=True)
        fold_scores = []
        for k, (tr_idx, va_idx) in enumerate(expanding_splits(sample_df, n_splits=3, horizon=8)):
            tr, va = sample_df.iloc[tr_idx], sample_df.iloc[va_idx]
            pipe = PerSeriesProphetForecaster(**cfg)
            pipe.fit(tr, tr['Weekly_Sales'])
            pred = pipe.predict(va)
            score = wmae(va['Weekly_Sales'], pred, va['IsHoliday'])
            fold_scores.append(score)
            wandb.log({'fold': k, 'fold_wmae': score})
        cv_mean = float(np.mean(fold_scores))
        wandb.log({'cv_wmae_mean': cv_mean, 'cv_wmae_std': float(np.std(fold_scores))})
        run.finish()
        trial_results.append({'config': cfg, 'cv_wmae_mean': cv_mean})
        cfg_tag = f"cp{cfg['changepoint_prior_scale']}_sp{cfg['seasonality_prior_scale']}_{cfg['seasonality_mode']}_hp{cfg['holidays_prior_scale']}"
        mlflow.log_metric(f'wmae_{cfg_tag}', cv_mean)
        print(f'config {i} {cfg}: CV WMAE={cv_mean:,.1f}')

    best_trial = min(trial_results, key=lambda t: t['cv_wmae_mean'])
    best_config = best_trial['config']
    mlflow.log_param('best_config', best_config)
    mlflow.log_metric('best_cv_wmae', best_trial['cv_wmae_mean'])

print('best config:', best_config, '| CV WMAE:', best_trial['cv_wmae_mean'])

## Run 3 results — visual comparison


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pathlib

cv_df = pd.DataFrame([
    {**t['config'], 'cv_wmae_mean': t['cv_wmae_mean']}
    for t in trial_results
]).sort_values('cv_wmae_mean').reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 10), gridspec_kw={'width_ratios': [1, 1.2]})

labels = [f"cp={r.changepoint_prior_scale:g} sp={r.seasonality_prior_scale:g} {r.seasonality_mode[:4]} hp={r.holidays_prior_scale:g}" for r in cv_df.itertuples()]
bar_colors = ['#c0392b' if i == 0 else '#5b8fd6' for i in range(len(cv_df))]
axes[0].barh(range(len(cv_df)), cv_df['cv_wmae_mean'], color=bar_colors)
axes[0].set_yticks(range(len(cv_df)))
axes[0].set_yticklabels(labels, fontsize=5.5)
axes[0].invert_yaxis()
axes[0].set_xlabel('CV WMAE (mean over 3 folds)')
axes[0].set_title(f'All {len(cv_df)} configs, sorted best -> worst')
axes[0].grid(axis='x', alpha=0.3)

best_mode = cv_df.iloc[0]['seasonality_mode']
best_hp = cv_df.iloc[0]['holidays_prior_scale']
subset = cv_df[(cv_df['seasonality_mode'] == best_mode) & (cv_df['holidays_prior_scale'] == best_hp)]
pivot = subset.pivot(index='changepoint_prior_scale', columns='seasonality_prior_scale', values='cv_wmae_mean')
sns.heatmap(pivot, annot=True, fmt=',.0f', cmap='RdYlGn_r', ax=axes[1], cbar_kws={'label': 'CV WMAE'})
axes[1].set_title(f'changepoint x seasonality prior scale\n@ {best_mode}, holidays_prior_scale={best_hp:g}')

plt.tight_layout()

cv_csv_path = pathlib.Path(WORKING_DIR) / 'prophet_cv_results.csv'
cv_df.to_csv(cv_csv_path, index=False)

with mlflow.start_run(run_name=f'{MODEL_NAME}_CV_Summary'):
    mlflow.log_figure(fig, 'cv_comparison.png')
    mlflow.log_artifact(str(cv_csv_path))

run = wandb.init(project=WANDB_PROJECT, group=MODEL_NAME, job_type='cv_summary', name=f'{MODEL_NAME}_CV_Summary')
wandb.log({'cv_comparison': wandb.Image(fig), 'cv_results_table': wandb.Table(dataframe=cv_df)})
run.finish()

plt.show()

## Run 4 — Final fit + save Pipeline


In [ ]:
run = wandb.init(project=WANDB_PROJECT, group=MODEL_NAME, job_type='final', name=f'{MODEL_NAME}_Final', config=best_config)

with mlflow.start_run(run_name=f'{MODEL_NAME}_Final') as mlflow_run:
    mlflow.log_params(best_config)

    holdout_tr, holdout_va = time_holdout(raw_train, weeks=8)
    p = PerSeriesProphetForecaster(**best_config)
    p.fit(raw_train.iloc[holdout_tr], raw_train.iloc[holdout_tr]['Weekly_Sales'])
    hv = raw_train.iloc[holdout_va]
    holdout_pred = p.predict(hv)
    holdout_wmae = wmae(hv['Weekly_Sales'], holdout_pred, hv['IsHoliday'])
    wandb.log({'holdout_wmae': holdout_wmae, 'n_failed_fits': p.n_failed_, 'n_series': p.n_series_})
    mlflow.log_metric('holdout_wmae', holdout_wmae)
    mlflow.log_metric('n_failed_fits', p.n_failed_)
    print(f'holdout WMAE: {holdout_wmae} | {p.n_failed_}/{p.n_series_} series failed to fit (fell back)')

    final_pipe = PerSeriesProphetForecaster(**best_config)
    final_pipe.fit(raw_train, raw_train['Weekly_Sales'])

    import pickle, pathlib
    from sklearn.pipeline import Pipeline as SkPipeline
    from mlflow.models import infer_signature

    out_dir = pathlib.Path(WORKING_DIR) / 'models' / MODEL_NAME
    out_dir.mkdir(parents=True, exist_ok=True)
    with open(out_dir / 'pipeline.pkl', 'wb') as f:
        pickle.dump(final_pipe, f)
    wandb.log_artifact(str(out_dir), name=f'{MODEL_NAME}_pipeline', type='model')

    sk_pipe = SkPipeline([('model', final_pipe)])
    example = raw_test.head(5)
    sig = infer_signature(example, final_pipe.predict(example))
    mlflow.sklearn.log_model(sk_pipe, artifact_path='pipeline', signature=sig, input_example=example, serialization_format='cloudpickle')
    mlflow.log_param('test_horizon', int(raw_test['Date'].nunique()))

run.finish()
print('saved pipeline to', out_dir, '| MLflow run:', mlflow_run.info.run_id)

## Run 4 results — holdout diagnostics


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(19, 10))

ax = axes[0, 0]
ax.scatter(hv['Weekly_Sales'], holdout_pred, s=8, alpha=0.25, color='#5b8fd6')
lims = [0, max(hv['Weekly_Sales'].max(), holdout_pred.max())]
ax.plot(lims, lims, 'r--', lw=1.5, label='perfect prediction')
ax.set_xlabel('Actual Weekly_Sales')
ax.set_ylabel('Predicted Weekly_Sales')
ax.set_title('Actual vs Predicted (holdout)')
ax.legend()
ax.grid(alpha=0.3)

ax = axes[0, 1]
residuals = hv['Weekly_Sales'].to_numpy() - holdout_pred
ax.hist(residuals, bins=60, color='#5b8fd6', edgecolor='black', alpha=0.8)
ax.axvline(0, color='r', linestyle='--', lw=1.5)
ax.set_xlabel('Residual (actual - predicted)')
ax.set_ylabel('Count')
ax.set_title(f'Residual distribution (mean={residuals.mean():,.0f}, std={residuals.std():,.0f})')
ax.grid(alpha=0.3)

ax = axes[0, 2]
agg = hv.assign(pred=holdout_pred).groupby('Date')[['Weekly_Sales', 'pred']].sum()
ax.plot(agg.index, agg['Weekly_Sales'], marker='o', label='Actual', color='#2c3e50')
ax.plot(agg.index, agg['pred'], marker='o', label='Predicted', color='#c0392b')
ax.set_xlabel('Date')
ax.set_ylabel('Total Weekly_Sales (all series)')
ax.set_title('Aggregate sales, actual vs predicted (holdout weeks)')
ax.legend()
ax.grid(alpha=0.3)
plt.setp(ax.get_xticklabels(), rotation=30, ha='right')

ax = axes[1, 0]
holiday_mask = hv['IsHoliday'].to_numpy().astype(bool)
wmae_holiday = wmae(hv['Weekly_Sales'][holiday_mask], holdout_pred[holiday_mask], hv['IsHoliday'][holiday_mask]) if holiday_mask.any() else float('nan')
wmae_non_holiday = wmae(hv['Weekly_Sales'][~holiday_mask], holdout_pred[~holiday_mask], hv['IsHoliday'][~holiday_mask])
bars = ax.bar(['Holiday weeks (w=5)', 'Non-holiday weeks (w=1)'], [wmae_holiday, wmae_non_holiday], color=['#c0392b', '#5b8fd6'])
ax.bar_label(bars, fmt='%.0f')
ax.axhline(holdout_wmae, color='black', linestyle='--', lw=1.5, label=f'Overall WMAE={holdout_wmae:,.0f}')
ax.set_ylabel('WMAE')
ax.set_title('Where the error is coming from')
ax.legend()
ax.grid(axis='y', alpha=0.3)

ax = axes[1, 1]
fitted_keys = [k for k, m in p.models_.items() if m is not None]
sample_key = max(fitted_keys, key=lambda k: p.last_date_[k])
sample_model = p.models_[sample_key]
hist = raw_train[(raw_train['Store'] == sample_key[0]) & (raw_train['Dept'] == sample_key[1])].sort_values('Date')
components = sample_model.predict(pd.DataFrame({'ds': hist['Date']}))
ax.plot(components['ds'], components['yhat'], label='yhat (trend + seasonality + holidays)', color='#c0392b', lw=1.2)
ax.plot(components['ds'], components['trend'], label='trend only', color='#2c3e50', lw=1.8)
for hol_dates, hol_name in [(SUPER_BOWL, None), (LABOR_DAY, None), (THANKSGIVING, None), (CHRISTMAS, None)]:
    in_range = hol_dates[(hol_dates >= hist['Date'].min()) & (hol_dates <= hist['Date'].max())]
    for d in in_range:
        ax.axvline(d, color='#27ae60', linestyle=':', lw=1, alpha=0.6)
ax.set_title(f'Prophet decomposition, Store {sample_key[0]} Dept {sample_key[1]}\n(green dotted = holiday weeks)')
ax.set_xlabel('Date')
ax.set_ylabel('Weekly_Sales')
ax.legend(fontsize=7)
ax.grid(alpha=0.3)
plt.setp(ax.get_xticklabels(), rotation=30, ha='right')

axes[1, 2].axis('off')

plt.tight_layout()

diag_run = mlflow.start_run(run_name=f'{MODEL_NAME}_Final_Diagnostics')
mlflow.log_figure(fig, 'holdout_diagnostics.png')
mlflow.log_metric('holdout_wmae_holiday', wmae_holiday)
mlflow.log_metric('holdout_wmae_non_holiday', wmae_non_holiday)
mlflow.end_run()

wandb_diag_run = wandb.init(project=WANDB_PROJECT, group=MODEL_NAME, job_type='final_diagnostics', name=f'{MODEL_NAME}_Final_Diagnostics')
wandb.log({'holdout_diagnostics': wandb.Image(fig), 'holdout_wmae': holdout_wmae, 'holdout_wmae_holiday': wmae_holiday, 'holdout_wmae_non_holiday': wmae_non_holiday})
wandb_diag_run.finish()

plt.show()